# RAG Config Comparison

Goal: compare different RAG chunking configurations using local LLM-as-judge evaluation.

We will test multiple `chunk_size` and `chunk_overlap` settings, run the same eval dataset through each setup, and compare:

- retrieval relevance
- correctness
- faithfulness


## Agenda

1. Load the PDF once.
2. Define the same eval dataset.
3. Build a reusable RAG pipeline for each chunk config.
4. Run RAG for each question and config.
5. Use Ollama/Gemma as a local judge.
6. Compute average scores per config.
7. Compare configs and decide which one is better for this dataset.


## Why This Is Production-Relevant

Production RAG changes should be measured, not guessed. If we change chunking, retrieval `k`, embedding model, prompt, or vector DB, we rerun evals and check whether quality improved or regressed.


In [2]:
from pathlib import Path
import json
import re
import time

import ollama

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS

MODEL_NAME = "gemma4:e4b"
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"


In [26]:
pdf_path = Path("../data/Understanding_Climate_Change.pdf")

loader = PyPDFLoader(str(pdf_path))
documents = loader.load()

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME
)

print("PDF exists:", pdf_path.exists())
print("Page documents:", len(documents))
print("Embedding model:", EMBEDDING_MODEL_NAME)

PDF exists: True
Page documents: 33
Embedding model: sentence-transformers/all-MiniLM-L6-v2


In [28]:
def build_rag_pipeline(chunk_size, chunk_overlap):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
    )

    chunks = splitter.split_documents(documents)
    vector_store = FAISS.from_documents(chunks, embedding_model)

    return {
        "chunk_size": chunk_size,
        "chunk_overlap": chunk_overlap,
        "chunks": chunks,
        "vector_store": vector_store,
    }



In [44]:
baseline_pipeline = build_rag_pipeline(
    chunk_size=1000,
    chunk_overlap=100,
)

print("Chunk size:", baseline_pipeline["chunk_size"]) 
print("Chunk overlap:", baseline_pipeline["chunk_overlap"])
print("Chunks:", len(baseline_pipeline["chunks"]))
print("Vector store:", type(baseline_pipeline["vector_store"]))

Chunk size: 1000
Chunk overlap: 100
Chunks: 97
Vector store: <class 'langchain_community.vectorstores.faiss.FAISS'>


In [33]:
# augmentation and Generation 

def run_rag(question, vector_store,k=3):
    retrieved_docs = vector_store.similarity_search(question, k=k)

    context = "\n\n".join(doc.page_content for doc in retrieved_docs)
    prompt = f""" 
    Answer the question using only the context below.
    If the answer is not in the context, say: "I don't know based on the provided context."

    Question:
    {question}

    Context:
    {context}

    Answer in one short sentence:
    """
    reponse = ollama.chat(
        model=MODEL_NAME,
        messages= [
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return {
        "question": question,
        "generated_answer": reponse.message.content,
        "context" : context,
        "retrieved_docs": retrieved_docs
    }


In [34]:
sample = run_rag(
    question="What is the main cause of recent climate change?",
    vector_store=baseline_pipeline["vector_store"],
    k=3,
)

print("Question:")
print(sample["question"])

print("\nGenerated answer:")
print(sample["generated_answer"])

print("\nContext preview:")
print(sample["context"][:800])

Question:
What is the main cause of recent climate change?

Generated answer:
The primary cause of recent climate change is the increase in greenhouse gases in the atmosphere.

Context preview:
change the amount of solar energy our planet receives. During the Holocene epoch, which 
began at the end of the last ice age, human societies flourished, but the industrial era has seen 
unprecedented changes. 
Modern Observations 
Modern scientific observations indicate a rapid increase in global temperatures, sea levels, 
and extreme weather events. The Intergovernmental Panel on Climate Change (IPCC) has 
documented these changes extensively. Ice core samples, tree rings, and ocean sediments 
provide a historical record that scientists use to understand past climate conditions and 
predict future trends. The evidence overwhelmingly shows that recent changes are primarily 
driven by human activities, particularly the emission of greenhouse gases. 
Chapter 2: Causes of Climate Change 
Gr


In [35]:
eval_dataset = [
    {
        "question": "What is the main cause of recent climate change?",
        "expected_answer": "The increase in greenhouse gases in the atmosphere, mainly from human activities such as burning fossil fuels and deforestation.",
    },
    {
        "question": "How does burning fossil fuels contribute to climate change?",
        "expected_answer": "Burning fossil fuels releases large amounts of carbon dioxide, increasing greenhouse gases and warming the climate.",
    },
    {
        "question": "What are some effects of climate change?",
        "expected_answer": "Effects include rising temperatures, melting ice, sea-level rise, extreme weather, droughts, flooding, and ocean acidification.",
    },
    {
        "question": "How does renewable energy help mitigate climate change?",
        "expected_answer": "Renewable energy sources produce little to no greenhouse gas emissions and reduce reliance on fossil fuels.",
    },
    {
        "question": "What is ocean acidification?",
        "expected_answer": "Ocean acidification is the increase in ocean acidity caused by higher CO2 levels dissolving into seawater.",
    },
]

print("Eval questions:", len(eval_dataset))

Eval questions: 5


In [36]:
def extract_json(text):
    match = re.search(r"\{.*\}", text, re.DOTALL)

    if not match:
        raise ValueError(f"No JSON object found in response:\n{text}")

    return json.loads(match.group(0))

In [48]:
def judge_correctness(question, expected_answer, generated_answer):
    prompt = f"""
        you are evaluating a RAG system.
        Score the correctness of the system on a scale 0-2:
        0- if the generated answer is incorrect 
        1- if the generated answer is partially correct
        2- if the generated answer is correct

        Question: {question}
        Expected answer :{expected_answer}
        Generated answer : {generated_answer}

        Return only valid JSON in this format:
        {{
            "score": 0,
            "reason":"provide a short explanation"
        }}
    """
    response = ollama.chat(
        model=MODEL_NAME,
        messages=[
            {
                "role": "User",
                "content": prompt
            }
        ]
    )

    return extract_json(response.message.content)


In [38]:
def judge_faithfulness(question, context, generated_answer):
    prompt = f"""
You are evaluating whether a RAG answer is supported by the retrieved context.

Score faithfulness on a 0-2 scale:
0 = answer is not supported by the context or contradicts it
1 = answer is partially supported by the context
2 = answer is clearly supported by the context

Question:
{question}

Retrieved context:
{context}

Generated answer:
{generated_answer}

Return only valid JSON in this exact format:
{{
  "score": 0,
  "reason": "short explanation"
}}
"""

    response = ollama.chat(
        model=MODEL_NAME,
        messages=[
            {"role": "user", "content": prompt}
        ],
    )

    return extract_json(response.message.content)

In [40]:
def judge_retrieval_relevance(question, context):
    prompt = f"""
You are evaluating retrieved context for a RAG system.

Score retrieval relevance on a 0-2 scale:
0 = retrieved context is irrelevant and does not help answer the question
1 = retrieved context is partially relevant but incomplete or noisy
2 = retrieved context clearly contains information needed to answer the question

Question:
{question}

Retrieved context:
{context}

Return only valid JSON in this exact format:
{{
  "score": 0,
  "reason": "short explanation"
}}
"""

    response = ollama.chat(
        model=MODEL_NAME,
        messages=[
            {"role": "user", "content": prompt}
        ],
    )

    return extract_json(response.message.content)

In [41]:
def evaluate_rag_result(item, vector_store, k=3):
    rag_result = run_rag(
        question=item["question"],
        vector_store=vector_store,
        k=k,
    )

    correctness = judge_correctness(
        question=item["question"],
        expected_answer=item["expected_answer"],
        generated_answer=rag_result["generated_answer"],
    )

    faithfulness = judge_faithfulness(
        question=item["question"],
        context=rag_result["context"],
        generated_answer=rag_result["generated_answer"],
    )

    retrieval_relevance = judge_retrieval_relevance(
        question=item["question"],
        context=rag_result["context"],
    )

    return {
        "question": item["question"],
        "expected_answer": item["expected_answer"],
        "generated_answer": rag_result["generated_answer"],
        "context": rag_result["context"],
        "retrieved_docs": rag_result["retrieved_docs"],
        "correctness": correctness,
        "faithfulness": faithfulness,
        "retrieval_relevance": retrieval_relevance,
    }

In [49]:
test_result = evaluate_rag_result(
    item=eval_dataset[0],
    vector_store=baseline_pipeline["vector_store"],
    k=3,
)

print("Question:")
print(test_result["question"])

print("\nGenerated answer:")
print(test_result["generated_answer"])

print("\nCorrectness:")
print(test_result["correctness"])

print("\nFaithfulness:")
print(test_result["faithfulness"])

print("\nRetrieval relevance:")
print(test_result["retrieval_relevance"])

Question:
What is the main cause of recent climate change?

Generated answer:
The primary cause of recent climate change is the increase in greenhouse gases in the atmosphere.

Correctness:
{'score': 1, 'reason': 'The generated answer correctly identifies the primary cause (increase in greenhouse gases) but fails to provide the crucial context and details regarding the sources of these emissions (human activities like burning fossil fuels or deforestation), which were included in the expected answer.'}

Faithfulness:
{'score': 2, 'reason': 'The context explicitly states under \'Chapter 2: Causes of Climate Change\' that "The primary cause of recent climate change is the increase in greenhouse gases in the atmosphere." This directly supports the generated answer.'}

Retrieval relevance:
{'score': 2, 'reason': "The context clearly states that 'The primary cause of recent climate change is the increase in greenhouse gases in the atmosphere,' directly answering the question."}


In [51]:
def run_eval_for_pipeline(pipeline, eval_dataset, k=3):
    results = []

    for i, item in enumerate(eval_dataset, start=1):
        print(f"Evaluating {i}/{len(eval_dataset)}: {item['question']}")

        start_time = time.time()

        result = evaluate_rag_result(
            item=item,
            vector_store=pipeline["vector_store"],
            k=k,
        )

        elapsed_time = time.time() - start_time

        result["chunk_size"] = pipeline["chunk_size"]
        result["chunk_overlap"] = pipeline["chunk_overlap"]
        result["num_chunks"] = len(pipeline["chunks"])
        result["k"] = k
        result["elapsed_time"] = elapsed_time

        results.append(result)

    return results

In [52]:
baseline_results = run_eval_for_pipeline(
    pipeline=baseline_pipeline,
    eval_dataset=eval_dataset,
    k=3,
)

print("Baseline eval complete:", len(baseline_results))

Evaluating 1/5: What is the main cause of recent climate change?
Evaluating 2/5: How does burning fossil fuels contribute to climate change?
Evaluating 3/5: What are some effects of climate change?
Evaluating 4/5: How does renewable energy help mitigate climate change?
Evaluating 5/5: What is ocean acidification?
Baseline eval complete: 5


In [53]:
def average_metric(results, metric_name):
    scores = []

    for result in results:
        score = result[metric_name]["score"]
        scores.append(score)

    return sum(scores) / len(scores)


def summarize_eval_results(results):
    summary = {
        "chunk_size": results[0]["chunk_size"],
        "chunk_overlap": results[0]["chunk_overlap"],
        "num_chunks": results[0]["num_chunks"],
        "k": results[0]["k"],
        "avg_correctness": average_metric(results, "correctness"),
        "avg_faithfulness": average_metric(results, "faithfulness"),
        "avg_retrieval_relevance": average_metric(results, "retrieval_relevance"),
        "avg_elapsed_time": sum(result["elapsed_time"] for result in results) / len(results),
    }

    return summary

In [54]:
chunk_configs = [
    {"chunk_size": 500, "chunk_overlap": 100},
    {"chunk_size": 1000, "chunk_overlap": 200},
    {"chunk_size": 1500, "chunk_overlap": 300},
    {"chunk_size": 2000, "chunk_overlap": 400},
]

In [55]:
all_config_summaries = []
all_config_results = {}

for config in chunk_configs:
    print("\n" + "=" * 80)
    print("Running config:", config)

    pipeline = build_rag_pipeline(
        chunk_size=config["chunk_size"],
        chunk_overlap=config["chunk_overlap"],
    )

    results = run_eval_for_pipeline(
        pipeline=pipeline,
        eval_dataset=eval_dataset,
        k=3,
    )

    summary = summarize_eval_results(results)

    all_config_summaries.append(summary)

    config_name = f"{config['chunk_size']}_{config['chunk_overlap']}"
    all_config_results[config_name] = results

print("Done running all configs.")


Running config: {'chunk_size': 500, 'chunk_overlap': 100}
Evaluating 1/5: What is the main cause of recent climate change?
Evaluating 2/5: How does burning fossil fuels contribute to climate change?
Evaluating 3/5: What are some effects of climate change?
Evaluating 4/5: How does renewable energy help mitigate climate change?
Evaluating 5/5: What is ocean acidification?

Running config: {'chunk_size': 1000, 'chunk_overlap': 200}
Evaluating 1/5: What is the main cause of recent climate change?


KeyboardInterrupt: 

In [ ]:
for summary in all_config_summaries:
    print("=" * 80)
    print(f"chunk_size: {summary['chunk_size']}")
    print(f"chunk_overlap: {summary['chunk_overlap']}")
    print(f"num_chunks: {summary['num_chunks']}")
    print(f"k: {summary['k']}")
    print(f"avg_correctness: {summary['avg_correctness']:.2f} / 2")
    print(f"avg_faithfulness: {summary['avg_faithfulness']:.2f} / 2")
    print(f"avg_retrieval_relevance: {summary['avg_retrieval_relevance']:.2f} / 2")
    print(f"avg_elapsed_time: {summary['avg_elapsed_time']:.2f}s")

In [ ]:
sorted_summaries = sorted(
    all_config_summaries,
    key=lambda item: (
        item["avg_correctness"],
        item["avg_faithfulness"],
        item["avg_retrieval_relevance"],
    ),
    reverse=True,
)

print("Best configs by correctness, faithfulness, retrieval relevance:")

for summary in sorted_summaries:
    print(
        f"chunk_size={summary['chunk_size']}, "
        f"overlap={summary['chunk_overlap']}, "
        f"chunks={summary['num_chunks']}, "
        f"correctness={summary['avg_correctness']:.2f}, "
        f"faithfulness={summary['avg_faithfulness']:.2f}, "
        f"retrieval={summary['avg_retrieval_relevance']:.2f}, "
        f"time={summary['avg_elapsed_time']:.2f}s"
    )